# Analyse tödlicher Verkehrsunfälle in Deutschland

## Teil 3: Visualisierung  Dashboards
### 1. Verkehrsunfälle nach Kategorie Balkendiagrame
### 2. Räumliche Verteilung der Verkehrsunfälle
### 3. Verkehrsunfälle nach Wochentag und Stunde

<a class="anchor" id="0.1"></a>

# Table of Contents 

1. Daten Laden
2. Layout
3. Callbacks


# 1. Daten Laden  <a class="anchor" id="1"></a>


In [1]:
import pandas as pd

df = pd.read_pickle(r'Data\ToedlicheUnfaelle_DE_J2025.pkl')
print(df.info())

<class 'pandas.DataFrame'>
Index: 2477 entries, 79 to 272949
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   UIDENTSTLAE         2477 non-null   object
 1   ULAND               2477 non-null   int64 
 2   UREGBEZ             2477 non-null   int64 
 3   UKREIS              2477 non-null   int64 
 4   UGEMEINDE           2477 non-null   int64 
 5   UJAHR               2477 non-null   int64 
 6   UMONAT              2477 non-null   int64 
 7   USTUNDE             2477 non-null   int64 
 8   UWOCHENTAG          2477 non-null   int64 
 9   UKATEGORIE          2477 non-null   int64 
 10  UART                2477 non-null   int64 
 11  UTYP1               2477 non-null   int64 
 12  ULICHTVERH          2477 non-null   int64 
 13  IstStrassenzustand  2477 non-null   int64 
 14  IstRad              2477 non-null   int64 
 15  IstPKW              2477 non-null   int64 
 16  IstFuss             2477 non-null   i

In [2]:
# ============================================================
# DASHBOARD: Verkehrsunfälle Deutschland 2025
# ============================================================

import pandas as pd
import geopandas as gpd
import plotly.express as px

from dash import Dash, dcc, html, Input, Output


# ============================================================
# 1. DATEN VORBEREITEN
# ============================================================

df_dash = df.copy()


# ============================================================
# 2. VERKEHRSTEILNEHMER
# ============================================================

verkehr_vars = {
    "Fußgänger": "IstFuss",
    "Radfahrer": "IstRad",
    "Pkw": "IstPKW",
    "Gkfz": "IstGkfz",
    "Krad": "IstKrad",
    "Sonstige": "IstSonstige"
}


# ============================================================
# 3. KATEGORIEN VORBEREITEN
# ============================================================

# ------------------------------------------------------------
# Wochentag
# ------------------------------------------------------------

wochentag_order = [
    "Montag",
    "Dienstag",
    "Mittwoch",
    "Donnerstag",
    "Freitag",
    "Samstag",
    "Sonntag"
]


# ------------------------------------------------------------
# Stunde
# ------------------------------------------------------------

df_dash["Stunde"] = df_dash["USTUNDE"]


# ------------------------------------------------------------
# Tageslicht
# ------------------------------------------------------------

licht_labels = {
    0: "Tageslicht",
    1: "Dämmerung",
    2: "Dunkelheit"
}

df_dash["Tageslicht"] = df_dash[
    "ULICHTVERH"
].map(licht_labels)


# ------------------------------------------------------------
# Monat
# ------------------------------------------------------------

monat_labels = {
    1: "Januar",
    2: "Februar",
    3: "März",
    4: "April",
    5: "Mai",
    6: "Juni",
    7: "Juli",
    8: "August",
    9: "September",
    10: "Oktober",
    11: "November",
    12: "Dezember"
}

df_dash["Monat"] = df_dash[
    "UMONAT"
].map(monat_labels)


# ------------------------------------------------------------
# Kategorien
# ------------------------------------------------------------

kategorien = {
    "Wochentag": "Wochentag",
    "Stunde": "Stunde",
    "Tageslicht": "Tageslicht",
    "Monat": "Monat",
    "Bundesland": "Bundesland"
}


# ============================================================
# 4. LOKALE DEUTSCHLANDKARTE LADEN
# ============================================================

germany = gpd.read_file(
    "data/germany.geojson"
).to_crs("EPSG:25832")


# ============================================================
# 5. DASH APP
# ============================================================

app = Dash(__name__)


# ============================================================
# 6. LAYOUT
# ============================================================

app.layout = html.Div(
    [

        # ====================================================
        # HAUPTTITEL
        # ====================================================

        html.H1(
            "Verkehrsunfälle Deutschland 2025",

            style={
                "textAlign": "center",
                "marginBottom": "20px"
            }
        ),

     html.P([
            "Projektarbeit, Kurs:",
    html.Strong("Data Analytics",style={"color": "blue"})," Dozent:",
    html.Strong(" Herr Axel Wemmel",style={"color": "blue"}), ", Teilnehmer:",
    html.Strong(" Dr. Abdelmoneim Desouki",style={"color": "blue"}),", Date:25/9/2026"],

            style={
                "textAlign": "left",
                "marginBottom": "40px"
            }
        ),
        # ====================================================
        # TEIL 1 – BALKENDIAGRAMM
        # ====================================================

        html.H2(
            "1. Verkehrsunfälle nach Kategorie",

            style={
                "marginBottom": "20px"
            }
        ),


        # ----------------------------------------------------
        # Dropdowns
        # ----------------------------------------------------

        html.Div(
            [

                # Verkehrsteilnehmer
                html.Div(
                    [

                        html.Label(
                            "Verkehrsteilnehmer:",

                            style={
                                "fontWeight": "bold"
                            }
                        ),

                        dcc.Dropdown(

                            id="bar-verkehrsteilnehmer",

                            options=[
                                {
                                    "label": "Alle",
                                    "value": "Alle"
                                }
                            ]
                            +
                            [
                                {
                                    "label": name,
                                    "value": var
                                }

                                for name, var
                                in verkehr_vars.items()
                            ],

                            value="Alle",

                            clearable=False
                        )

                    ],

                    style={
                        "width": "40%",
                        "display": "inline-block",
                        "marginRight": "5%"
                    }
                ),


                # Kategorie
                html.Div(
                    [

                        html.Label(
                            "Kategorie:",

                            style={
                                "fontWeight": "bold"
                            }
                        ),

                        dcc.Dropdown(

                            id="bar-kategorie",

                            options=[
                                {
                                    "label": name,
                                    "value": name
                                }

                                for name in kategorien
                            ],

                            value="Bundesland",

                            clearable=False
                        )

                    ],

                    style={
                        "width": "40%",
                        "display": "inline-block"
                    }
                )

            ],

            style={
                "width": "90%",
                "margin": "auto",
                "marginBottom": "20px"
            }
        ),


        # ----------------------------------------------------
        # Balkendiagramm
        # ----------------------------------------------------

        dcc.Graph(

            id="barplot",

            style={
                "width": "95%",
                "height": "650px",
                "margin": "auto"
            }

        ),


        # ====================================================
        # TRENNLINIE
        # ====================================================

        html.Hr(

            style={
                "marginTop": "50px",
                "marginBottom": "40px"
            }

        ),


        # ====================================================
        # TEIL 2 – KARTE
        # ====================================================

        html.H2(

            "2. Räumliche Verteilung der Verkehrsunfälle",

            style={
                "marginBottom": "20px"
            }

        ),


        # ----------------------------------------------------
        # CHECKBOXEN + KARTE
        # ----------------------------------------------------

        html.Div(
            [

                # ==================================================
                # CHECKBOXEN – LINKER BEREICH
                # ==================================================

                html.Div(
                    [

                        html.Label(

                            "Verkehrsteilnehmer:",

                            style={
                                "fontWeight": "bold",
                                "display": "block",
                                "marginBottom": "20px"
                            }

                        ),


                        dcc.Checklist(

                            id="map-verkehrsteilnehmer",

                            options=[
                                {
                                    "label": name,
                                    "value": var
                                }

                                for name, var
                                in verkehr_vars.items()
                            ],

                            value=list(
                                verkehr_vars.values()
                            ),

                            # Vertikal
                            labelStyle={
                                "display": "block",
                                "marginBottom": "15px"
                            },

                            inputStyle={
                                "marginRight": "8px"
                            }

                        )

                    ],

                    style={
                        "flex": "0 0 180px",
                        "padding": "20px 10px",
                        "boxSizing": "border-box"
                    }

                ),


                # ==================================================
                # KARTE – RECHTER BEREICH
                # ==================================================

                html.Div(
                    [

                        dcc.Graph(

                            id="map",

                            style={
                                "width": "600px",
                                "height": "800px"
                            },

                            config={
                                "responsive": False
                            }

                        )

                    ],

                    style={
                        "flex": "0 0 600px",
                        "width": "600px",
                        "height": "800px"
                    }

                )

            ],

            style={
                "display": "flex",
                "flexDirection": "row",
                "width": "800px",
                "margin": "auto",
                "alignItems": "flex-start"
            }

        ),


        # ====================================================
        # TRENNLINIE
        # ====================================================

        html.Hr(

            style={
                "marginTop": "50px",
                "marginBottom": "40px"
            }

        ),


        # ====================================================
        # TEIL 3 – WOCHENTAG / STUNDE HEATMAP
        # ====================================================

        html.H2(

            "3. Verkehrsunfälle nach Wochentag und Stunde",

            style={
                "marginBottom": "20px"
            }

        ),


        # ----------------------------------------------------
        # MONAT SLIDER
        # ----------------------------------------------------

        html.Div(
            [

                html.Label(

                    "Monat:",

                    style={
                        "fontWeight": "bold",
                        "display": "block",
                        "marginBottom": "10px"
                    }

                ),


                dcc.RangeSlider(

                    id="heatmap-monat",

                    min=1,

                    max=12,

                    step=1,

                    value=[
                        1,
                        12
                    ],

                    marks={

                        1: "Jan",
                        2: "Feb",
                        3: "Mär",
                        4: "Apr",
                        5: "Mai",
                        6: "Jun",
                        7: "Jul",
                        8: "Aug",
                        9: "Sep",
                        10: "Okt",
                        11: "Nov",
                        12: "Dez"

                    },

                    allowCross=False,

                    tooltip={

                        "placement": "bottom",

                        "always_visible": True

                    }

                )

            ],

            style={
                "width": "80%",
                "margin": "auto",
                "marginBottom": "40px"
            }

        ),


        # ----------------------------------------------------
        # HEATMAP
        # ----------------------------------------------------

        dcc.Graph(

            id="wochentag-stunde-heatmap",

            style={
                "width": "900px",
                "height": "650px",
                "margin": "auto"
            }

        )

    ],


    # ========================================================
    # GLOBALER STYLE
    # ========================================================

    style={
        "fontFamily": "Arial",
        "padding": "20px"
    }

)


# ============================================================
# 7. CALLBACK – BALKENDIAGRAMM
# ============================================================

@app.callback(

    Output(
        "barplot",
        "figure"
    ),

    Input(
        "bar-verkehrsteilnehmer",
        "value"
    ),

    Input(
        "bar-kategorie",
        "value"
    )

)
def update_barplot(

    verkehrsteilnehmer,

    kategorie

):

    # --------------------------------------------------------
    # Daten filtern
    # --------------------------------------------------------

    if verkehrsteilnehmer == "Alle":

        data = df_dash.copy()

        teilnehmer_name = "Alle"

    else:

        data = df_dash[
            df_dash[verkehrsteilnehmer] == 1
        ].copy()

        teilnehmer_name = next(

            name

            for name, var
            in verkehr_vars.items()

            if var == verkehrsteilnehmer

        )


    # --------------------------------------------------------
    # Kategorie bestimmen
    # --------------------------------------------------------

    column = kategorien[
        kategorie
    ]


    # --------------------------------------------------------
    # Anzahl berechnen
    # --------------------------------------------------------

    counts = (

        data[column]

        .value_counts()

        .reset_index()

    )


    counts.columns = [

        column,

        "Anzahl"

    ]


    # --------------------------------------------------------
    # Sortierung
    # --------------------------------------------------------

    if kategorie == "Wochentag":

        counts[column] = pd.Categorical(

            counts[column],

            categories=wochentag_order,

            ordered=True

        )

        counts = counts.sort_values(
            column
        )


    elif kategorie == "Stunde":

        counts = counts.sort_values(
            column
        )


    elif kategorie == "Monat":

        counts[column] = pd.Categorical(

            counts[column],

            categories=list(
                monat_labels.values()
            ),

            ordered=True

        )

        counts = counts.sort_values(
            column
        )


    elif kategorie == "Tageslicht":

        licht_order = [

            "Tageslicht",

            "Dämmerung",

            "Dunkelheit"

        ]

        counts[column] = pd.Categorical(

            counts[column],

            categories=licht_order,

            ordered=True

        )

        counts = counts.sort_values(
            column
        )


    # --------------------------------------------------------
    # Balkendiagramm
    # --------------------------------------------------------

    fig = px.bar(

        counts,

        x=column,

        y="Anzahl",

        text="Anzahl",

        title=(

            f"Verkehrsunfälle nach {kategorie}<br>"

            f"Verkehrsteilnehmer: {teilnehmer_name}"

        ),

        labels={

            column: kategorie,

            "Anzahl": "Anzahl Unfälle"

        }

    )


    # --------------------------------------------------------
    # Hover
    # --------------------------------------------------------

    fig.update_traces(

        textposition="outside",

        hovertemplate=(

            "<b>%{x}</b><br>"

            "Anzahl Unfälle: %{y}"

            "<extra></extra>"

        )

    )


    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    fig.update_layout(

        title_x=0.5,

        xaxis_title=kategorie,

        yaxis_title="Anzahl Unfälle",

        template="plotly_white"

    )


    return fig


# ============================================================
# 8. CALLBACK – KARTE
# ============================================================

@app.callback(

    Output(
        "map",
        "figure"
    ),

    Input(
        "map-verkehrsteilnehmer",
        "value"
    )

)
def update_map(

    selected_vars

):

    # ========================================================
    # Keine Auswahl
    # ========================================================

    if not selected_vars:

        fig = px.scatter_map(

            zoom=5.5,

            center={

                "lat": 51.2,

                "lon": 10.4

            }

        )


        fig.update_layout(

            map_style="open-street-map",

            title=(
                "Keine Verkehrsteilnehmer ausgewählt"
            )

        )


        return fig


    # ========================================================
    # Daten filtern
    # ========================================================

    # Ein Unfall wird berücksichtigt,
    # wenn mindestens eine ausgewählte
    # Verkehrsteilnehmer-Variable = 1 ist.

    mask = df_dash[
        selected_vars
    ].any(
        axis=1
    )


    data = df_dash[
        mask
    ].copy()


    # ========================================================
    # Koordinaten vorbereiten
    # ========================================================

    for col in [

        "LINREFX",

        "LINREFY"

    ]:

        data[col] = (

            data[col]

            .astype(str)

            .str.replace(

                ",",

                ".",

                regex=False

            )

            .astype(float)

        )


    data = data.dropna(

        subset=[

            "LINREFX",

            "LINREFY"

        ]

    )


    # ========================================================
    # GeoDataFrame
    # ========================================================

    gdf = gpd.GeoDataFrame(

        data,

        geometry=gpd.points_from_xy(

            data["LINREFX"],

            data["LINREFY"]

        ),

        crs="EPSG:25832"

    )


    # ========================================================
    # Nur Punkte innerhalb Deutschlands
    # ========================================================

    gdf = gpd.clip(

        gdf,

        germany

    )


    # ========================================================
    # WGS84 für Plotly
    # ========================================================

    gdf = gdf.to_crs(

        "EPSG:4326"

    )


    # ========================================================
    # Explizite Koordinatenspalten
    # ========================================================

    gdf["latitude"] = (
        gdf.geometry.y
    )

    gdf["longitude"] = (
        gdf.geometry.x
    )


    # ========================================================
    # Namen der ausgewählten Verkehrsteilnehmer
    # ========================================================

    selected_names = [

        name

        for name, var
        in verkehr_vars.items()

        if var in selected_vars

    ]


    selected_text = ", ".join(

        selected_names

    )


    # ========================================================
    # Karte
    # ========================================================

    fig = px.scatter_map(

        gdf,

        lat="latitude",

        lon="longitude",

        hover_name="Bundesland",

        hover_data={

            "latitude": False,

            "longitude": False,

            "Bundesland": True,

            "ULAND": True,

            "USTUNDE": True,

            "UMONAT": True

        },

        zoom=5.5,

        center={

            "lat": 51.2,

            "lon": 10.4

        },

        height=800,

        width=600,

        title=(

            "Räumliche Verteilung der Verkehrsunfälle<br>"

            f"Verkehrsteilnehmer: {selected_text}"

        )

    )


    # ========================================================
    # Unfallpunkte
    # ========================================================

    fig.update_traces(

        marker={

            "size": 6,

            "opacity": 0.6

        }

    )


    # ========================================================
    # Lokale Deutschlandgrenze
    # ========================================================

    germany_wgs84 = germany.to_crs(

        "EPSG:4326"

    )


    # ========================================================
    # Deutschlandgrenze zeichnen
    # ========================================================

    for geometry in germany_wgs84.geometry:


        # ----------------------------------------------------
        # Polygon
        # ----------------------------------------------------

        if geometry.geom_type == "Polygon":

            x, y = geometry.exterior.xy


            fig.add_scattermap(

                lon=list(x),

                lat=list(y),

                mode="lines",

                line={
                    "width": 2
                },

                hoverinfo="skip",

                showlegend=False

            )


        # ----------------------------------------------------
        # MultiPolygon
        # ----------------------------------------------------

        elif geometry.geom_type == "MultiPolygon":

            for polygon in geometry.geoms:

                x, y = polygon.exterior.xy


                fig.add_scattermap(

                    lon=list(x),

                    lat=list(y),

                    mode="lines",

                    line={
                        "width": 2
                    },

                    hoverinfo="skip",

                    showlegend=False

                )


    # ========================================================
    # Kartenlayout
    # ========================================================

    fig.update_layout(

        title_x=0.5,

        map_style="open-street-map",

        margin={

            "l": 0,

            "r": 0,

            "t": 70,

            "b": 0

        }

    )


    return fig


# ============================================================
# 9. CALLBACK – WOCHENTAG / STUNDE HEATMAP
# ============================================================

@app.callback(

    Output(

        "wochentag-stunde-heatmap",

        "figure"

    ),

    Input(

        "heatmap-monat",

        "value"

    )

)
def update_heatmap(

    monat_range

):

    # ========================================================
    # Monatbereich
    # ========================================================

    monat_min = monat_range[0]

    monat_max = monat_range[1]


    # ========================================================
    # Daten nach Monat filtern
    # ========================================================

    data = df_dash[

        df_dash["UMONAT"].between(

            monat_min,

            monat_max

        )

    ].copy()


    # ========================================================
    # Gruppieren
    # ========================================================

    heatmap_data = (

        data

        .groupby(

            [

                "Wochentag",

                "USTUNDE"

            ]

        )

        .size()

        .reset_index(

            name="Anzahl"

        )

    )


    # ========================================================
    # Vollständige Kombinationen
    #
    # Dadurch werden auch Kombinationen
    # mit 0 Unfällen angezeigt.
    # ========================================================

    complete_index = (

        pd.MultiIndex.from_product(

            [

                wochentag_order,

                list(range(24))

            ],

            names=[

                "Wochentag",

                "USTUNDE"

            ]

        )

    )


    heatmap_data = (

        heatmap_data

        .set_index(

            [

                "Wochentag",

                "USTUNDE"

            ]

        )

        .reindex(

            complete_index,

            fill_value=0

        )

        .reset_index()

    )


    # ========================================================
    # Pivot
    # ========================================================

    heatmap = heatmap_data.pivot(

        index="Wochentag",

        columns="USTUNDE",

        values="Anzahl"

    )


    # ========================================================
    # Heatmap
    # ========================================================

    fig = px.imshow(

        heatmap,

        x=list(range(24)),

        y=wochentag_order,

        text_auto=True,

        aspect="auto",

        labels={

            "x": "Stunde",

            "y": "Wochentag",

            "color": "Anzahl Unfälle"

        },

        title=(

            "Verkehrsunfälle nach Wochentag und Stunde<br>"

            f"Monat: {monat_min}–{monat_max}"

        )

    )


    # ========================================================
    # Heatmap Layout
    # ========================================================

    fig.update_layout(

        title_x=0.5,

        xaxis={

            "tickmode": "linear",

            "dtick": 1,

            "title": "Stunde"

        },

        yaxis={

            "title": "Wochentag"

        },

        template="plotly_white"

    )


    return fig


# ============================================================
# 10. APP STARTEN
# ============================================================

if __name__ == "__main__":

    app.run(
        debug=True
    )

## Mittwoch 17 Uhr gefährlichste Stunden dann Freitag 14 Uhr 
## Verkehrsteilnehmer - Monat : Fußgänger gefährlichste Zeit Dez und Jan , anderes von Gesamt, vermütlich wegen langezeit Dunkelheit in diese Zwei Monate